# Vision Transformer (ViT) Transfer Learning — CIFAR-10

## 이 노트북이 하는 일
Google 이 ImageNet-21k 에서 **사전학습한 ViT** (google/vit-base-patch16-224-in21k) 를 불러와서
**CIFAR-10** (10개 클래스 이미지 분류) 에 전이학습시키는 예제.

## ViT 의 핵심 아이디어 (왜 "Vision Transformer" 인가)
- 원래 Transformer 는 NLP 용 (토큰 시퀀스)
- ViT 는 **이미지를 16×16 패치로 쪼개서 토큰 시퀀스처럼 취급**
- 각 패치를 flatten + Linear 투영 → "이미지 패치 embedding" 이 됨
- 여기에 CLS 토큰 + positional embedding 을 더해서 기존 Transformer Encoder 에 그대로 투입
- 최종 CLS 토큰 출력으로 분류

이 방식이 ConvNet 보다 충분한 데이터가 있을 때 **더 잘 스케일** 한다는 게 핵심 발견.

## 왜 전이학습인가
- ViT 를 scratch 에서 학습하려면 수백만~수억 장의 이미지 필요 (ConvNet 보다 inductive bias 약해서)
- CIFAR-10 은 50k 장밖에 없음 → scratch 학습하면 잘 안 됨
- **ImageNet-21k 로 pre-train 된 모델을 가져와서 classifier head 만 바꾸고 fine-tuning** → 소량 데이터로도 좋은 성능

## 두 개의 셀
1. **[A]** `ViTImageProcessor` 사용 버전 (구버전 HuggingFace API)
2. **[B]** `AutoImageProcessor` + prefetch 등 개선 버전

[A] 는 동작 예시, [B] 는 더 깔끔한 파이프라인. 둘 다 같은 태스크를 수행.


---
## [0] 패키지 설치

- `transformers` : HuggingFace 의 사전학습 모델 허브 라이브러리
- `datasets` : HuggingFace 의 데이터셋 로더
- `-U datasets fsspec` : 최신 버전으로 업그레이드 (호환성 이슈 방지)


In [ ]:
# [0] 필요한 패키지 설치
!pip install -q transformers datasets
!pip install -U datasets fsspec


---
# [A] ViTImageProcessor 버전 (구 API)

### 왜 `TFViTForImageClassification` + `from_pt=True`
- `TFViTForImageClassification` : TensorFlow 용 ViT 분류 모델 래퍼 (내부는 `vit` backbone + `classifier` head)
- `from_pt=True` : HuggingFace 허브에 **PyTorch 체크포인트만** 있는 경우, TensorFlow 쪽으로 자동 변환 로딩
- `num_labels=10` : CIFAR-10 의 10 클래스에 맞춰서 **classifier head 를 새로 초기화**

### 왜 `model.vit.trainable = False` 로 백본을 얼리는가
- 백본 (큰 ViT) 은 이미 ImageNet-21k 로 충분히 학습됨 → 건드리지 않고 **feature extractor** 로만 사용
- classifier head (작은 Dense 레이어) 만 학습 → 학습 속도 빠름, 과적합 위험 ↓
- 나중에 fine-tuning 하려면 이 줄만 주석 처리하면 됨 (주석에 명시되어 있음)

### 왜 1000 / 200 개만 샘플링하는가
- `select(range(1000))` — 빠른 데모용 서브셋
- 실전에서는 전체 50k 장을 써야 제대로 된 성능


## [A-1] 데이터 + 전처리 + 모델 로드


In [ ]:
# [A-1] ViT 전이학습 (ViTImageProcessor 버전)
import tensorflow as tf
from transformers import TFViTForImageClassification, ViTImageProcessor
from datasets import load_dataset
import numpy as np
import matplotlib.pyplot as plt

# 1. 데이터셋 로드 — HuggingFace 허브에서 CIFAR-10 받아옴
dataset = load_dataset("cifar10")

# 2. Feature extractor (이미지 → ViT 입력 텐서로 변환: 리사이즈, 정규화 등)
processor = ViTImageProcessor.from_pretrained("google/vit-base-patch16-224-in21k")

# 3. 전처리 함수 정의 — dataset.map 으로 이미지마다 호출됨
def transform(example):
    # processor 는 이미지를 224x224 로 리사이즈하고 채널 정규화한 ndarray 반환
    inputs = processor(images=example["img"], return_tensors="np")
    inputs["label"] = example["label"]
    return inputs

# 4. 소규모 샘플만 사용 (속도 데모). 실전은 전체 50k 사용.
train_data = dataset["train"].select(range(1000)).map(transform)
val_data   = dataset["test"].select(range(200)).map(transform)

# 5. TF Dataset 변환
#    - pixel_values: (N, 3, 224, 224) 텐서로 stack
#    - label: 정수 레이블 리스트
#    - batch(16): 메모리 고려해서 작은 배치
def to_tf_dataset(data):
    return tf.data.Dataset.from_tensor_slices((
        {"pixel_values": np.stack([x["pixel_values"][0] for x in data])},
        [x["label"] for x in data],
    )).batch(16)

train_tfds = to_tf_dataset(train_data)
val_tfds   = to_tf_dataset(val_data)

# 6. ViT 모델 로드 — PyTorch 체크포인트를 TF 로 변환하며 CIFAR-10 용 10-class head 재초기화
model = TFViTForImageClassification.from_pretrained(
    "google/vit-base-patch16-224-in21k",
    num_labels=10,
    from_pt=True,                    # HF 허브에 TF 체크포인트가 없는 경우 자동 변환
)

# ★ 전이학습 핵심: 백본 freeze. classifier head 만 학습.
# 아래 줄을 지우면 백본까지 fine-tuning (더 좋은 성능이지만 느리고 overfit 위험↑)
model.vit.trainable = False

# 7. 컴파일 & 학습
# - learning_rate=5e-5: ViT fine-tuning 표준 범위 (너무 크면 사전학습된 표현을 망가뜨림)
# - SparseCategoricalCrossentropy(from_logits=True): 모델이 logits 을 반환하므로 softmax 는 loss 가 담당
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=5e-5),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"],
)
history = model.fit(train_tfds, validation_data=val_tfds, epochs=3)

# 8. 정확도 시각화
plt.plot(history.history["accuracy"], label="Train Acc")
plt.plot(history.history["val_accuracy"], label="Val Acc")
plt.xlabel("Epoch"); plt.ylabel("Accuracy")
plt.title("ViT (Hugging Face) on CIFAR-10")
plt.legend(); plt.grid(True)
plt.show()


---
# [B] AutoImageProcessor + prefetch 버전 (더 깔끔한 파이프라인)

### [A] 대비 개선점
1. **`AutoImageProcessor`** — 모델 이름만 주면 맞는 processor 를 자동 선택. 이식성 ↑
2. **`prefetch(tf.data.AUTOTUNE)`** — GPU 학습 중 CPU 가 다음 배치를 미리 준비 → 파이프라인 wait 감소
3. **`shuffle=True`** (train 에만) — 매 epoch 데이터 순서를 섞어서 일반화 ↑
4. **백본 freeze 를 주석 처리** — 기본은 전체 fine-tuning (성능↑). freeze 하고 싶으면 주석 해제.


## [B-1] TensorFlow + Transformers (TFViT) 버전


In [ ]:
# [B-1] 개선된 ViT 전이학습
import tensorflow as tf
from transformers import TFViTForImageClassification, AutoImageProcessor
from datasets import load_dataset
import numpy as np
import matplotlib.pyplot as plt

# 1) 데이터셋 로드 (CIFAR-10)
dataset = load_dataset("cifar10")

# 2) Feature extractor — AutoImageProcessor 가 모델 이름 보고 알맞은 클래스 선택
processor = AutoImageProcessor.from_pretrained("google/vit-base-patch16-224-in21k")

# 3) 전처리 함수 — 이미지 1장을 (3, 224, 224) 로 변환
def transform(example):
    proc = processor(images=example["img"], return_tensors="np")
    return {"pixel_values": proc["pixel_values"][0], "label": example["label"]}

# 4) 소규모 샘플 (속도 데모)
train_small = dataset["train"].select(range(1000)).map(transform)
val_small   = dataset["test"].select(range(200)).map(transform)

# 5) TF Dataset 변환 — prefetch 추가로 CPU↔GPU 파이프라인 최적화
def to_tfds(data, batch_size=16, shuffle=False):
    x = np.stack([r["pixel_values"] for r in data])
    y = np.array([r["label"] for r in data], dtype=np.int64)
    ds = tf.data.Dataset.from_tensor_slices(({"pixel_values": x}, y))
    if shuffle:
        # 매 epoch 다른 순서로 보도록 — train 일반화에 도움
        ds = ds.shuffle(len(y), reshuffle_each_iteration=True)
    # prefetch(AUTOTUNE): 학습 중 다음 배치를 미리 준비 → idle time 감소
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

train_tfds = to_tfds(train_small, shuffle=True)
val_tfds   = to_tfds(val_small)

# 6) 모델 로드 (from_pt=True — PyTorch 체크포인트를 TF 로)
model = TFViTForImageClassification.from_pretrained(
    "google/vit-base-patch16-224-in21k",
    num_labels=10,
    from_pt=True,
)

# (옵션) 백본만 freeze 하고 싶다면 아래 두 줄 주석 해제
# model.vit.trainable = False
# print("Backbone trainable?", model.vit.trainable, "| Classifier trainable?", model.classifier.trainable)

# 7) 컴파일 & 학습 — 전체 fine-tuning (백본 trainable=True 기본값)
#    learning_rate 5e-5 유지 — 전체 fine-tuning 이라도 사전학습된 가중치를 '살살' 조정
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=5e-5),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"],
)
history = model.fit(train_tfds, validation_data=val_tfds, epochs=3)

# 8) 정확도 시각화
plt.plot(history.history["accuracy"], label="Train Acc")
plt.plot(history.history["val_accuracy"], label="Val Acc")
plt.xlabel("Epoch"); plt.ylabel("Accuracy")
plt.title("TF ViT (HF) on CIFAR-10")
plt.legend(); plt.grid(True)
plt.show()
